# 01 – Limpeza e preparação dos dados de criminalidade (IPARDES)

Este notebook realiza a limpeza e a padronização das tabelas brutas de criminalidade
do IPARDES para o estado do Paraná (2018–2024). O objetivo é transformar os arquivos
originais, que estão em formato tabular complexo, em uma base única e organizada no
formato "longo", adequada para análise exploratória no notebook 02.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# pasta do notebook = /notebooks
BASE_DIR = Path("..")  # volta para a raiz do projeto
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR


(WindowsPath('../data/raw'), WindowsPath('../data/processed'))

In [ ]:
def carregar_tabela_ipardes(caminho_arquivo: Path, nome_fonte: str) -> pd.DataFrame:
    """
    Lê uma tabela do IPARDES no formato 2018-2024 e devolve
    um DataFrame no formato longo:
    municipio, regiao, ano, tipo_crime, quantidade, fonte
    """
    # 1) Leitura com encoding e separador corretos
    df = pd.read_csv(caminho_arquivo, sep="\t", encoding="utf-16")
    
    # 2) Renomear as duas primeiras colunas para algo útil
    df = df.rename(columns={
        df.columns[0]: "municipio",
        df.columns[1]: "regiao"
    })
    
    # 3) Cabeçalho "duplo": linha 0 = tipo de crime, linha 1 = ano
    categorias = df.iloc[0, 2:]   # do 3º em diante
    anos = df.iloc[1, 2:]
    
    # Criar um MultiIndex (tipo_crime, ano) para as colunas de valores
    multi_cols = pd.MultiIndex.from_arrays(
        [categorias.values, anos.values],
        names=["tipo_crime", "ano"]
    )
    
    # 4) Separar valores (a partir da linha 2) e metadados (município / região)
    valores = df.iloc[2:, 2:].copy()
    valores.columns = multi_cols  # aplica o MultiIndex
    
    # importante: manter o índice ORIGINAL (2..401) para casar com os municípios
    meta = df.loc[2:, ["municipio", "regiao"]].copy()
    
    # 5) Função para transformar strings em números
    def to_num(x):
        if isinstance(x, str):
            x = x.strip()
            if x in ("", "-"):
                return np.nan
            # tira separador de milhar e ajusta decimal
            x = x.replace(".", "").replace(",", ".")
            try:
                return float(x)
            except ValueError:
                return np.nan
        return x
    
    valores = valores.applymap(to_num)
    
    # 6) Wide -> Long (empilhar tipo_crime e ano)
    long_vals = valores.stack(["tipo_crime", "ano"]).reset_index()
    # level_0 é o índice da linha original (2..401)
    long_vals = long_vals.rename(columns={"level_0": "idx", 0: "quantidade"})
    
    # 7) Juntar com os metadados usando o índice original
    meta_idx = meta.reset_index().rename(columns={"index": "idx"})
    base = long_vals.merge(meta_idx, on="idx", how="left").drop(columns="idx")
    
    # 8) Ajustes finais
    base["ano"] = base["ano"].astype(int)
    base["fonte"] = nome_fonte
    
    # Ordenar colunas de forma amigável
    cols = ["municipio", "regiao", "ano", "tipo_crime", "quantidade", "fonte"]
    return base[cols]


In [4]:
caminho_crimes = RAW_DIR / "Crimes_Tabela_2018_a_2024.csv"
caminho_mvi = RAW_DIR / "Mortes Violentas Intencionais_Tabela_2018_a_2024.csv"

base_crimes = carregar_tabela_ipardes(caminho_crimes, "Crimes gerais")
base_mvi = carregar_tabela_ipardes(caminho_mvi, "Mortes violentas")

base_crimes.head(), base_crimes.shape, base_mvi.shape


C:\Users\guilh\AppData\Local\Temp\ipykernel_9532\2900863995.py:47: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  valores = valores.applymap(to_num)
C:\Users\guilh\AppData\Local\Temp\ipykernel_9532\2900863995.py:50: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long_vals = valores.stack(["tipo_crime", "ano"]).reset_index()
C:\Users\guilh\AppData\Local\Temp\ipykernel_9532\2900863995.py:47: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  valores = valores.applymap(to_num)
C:\Users\guilh\AppData\Local\Temp\ipykernel_9532\2900863995.py:50: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. S

(          municipio regiao   ano                 tipo_crime  quantidade  \
 0  Estado do Paraná      -  2018  Armas de Fogo Apreendidas      6267.0   
 1  Estado do Paraná      -  2019  Armas de Fogo Apreendidas      6150.0   
 2  Estado do Paraná      -  2020  Armas de Fogo Apreendidas      7129.0   
 3  Estado do Paraná      -  2021  Armas de Fogo Apreendidas      6666.0   
 4  Estado do Paraná      -  2022  Armas de Fogo Apreendidas      6216.0   
 
            fonte  
 0  Crimes gerais  
 1  Crimes gerais  
 2  Crimes gerais  
 3  Crimes gerais  
 4  Crimes gerais  ,
 (31212, 6),
 (2825, 6))

In [5]:
base_criminalidade = pd.concat([base_crimes, base_mvi], ignore_index=True)

base_criminalidade.to_csv(
    PROCESSED_DIR / "base_criminalidade_pr_limpa.csv",
    index=False
)

base_criminalidade.head(), base_criminalidade.shape


(          municipio regiao   ano                 tipo_crime  quantidade  \
 0  Estado do Paraná      -  2018  Armas de Fogo Apreendidas      6267.0   
 1  Estado do Paraná      -  2019  Armas de Fogo Apreendidas      6150.0   
 2  Estado do Paraná      -  2020  Armas de Fogo Apreendidas      7129.0   
 3  Estado do Paraná      -  2021  Armas de Fogo Apreendidas      6666.0   
 4  Estado do Paraná      -  2022  Armas de Fogo Apreendidas      6216.0   
 
            fonte  
 0  Crimes gerais  
 1  Crimes gerais  
 2  Crimes gerais  
 3  Crimes gerais  
 4  Crimes gerais  ,
 (34037, 6))